In [ ]:
import collections
import dataclasses
import enum
import importlib
import typing

import deepdiff
import matplotlib.pyplot as plt

import pyine.data.lmdb_io
import pyine.utils.code_blocks
import pyine.utils.code_exec
import pyine.utils.portability

In [ ]:
importlib.reload(pyine.data.lmdb_io)
importlib.reload(pyine.utils.code_exec)
importlib.reload(pyine.utils.code_blocks)
importlib.reload(pyine.utils.portability)

In [ ]:
parser = pyine.data.lmdb_io.LMDBReader(
    path="../data/2025-03-31-v01-lmdb",
)
target_difficulties = ["EASY"]
problem_count = len(parser)
print(f"{problem_count=}")
metadata = parser.get_metadata()
for metadata_key, metadata_value in metadata.items():
    print(f"{metadata_key}: {metadata_value}")

In [ ]:
difficulty_counts = collections.Counter()
for problem_idx, problem_data in enumerate(parser):
    difficulty_counts[problem_data["difficulty"]] += 1
print("Difficulty distribution:", dict(difficulty_counts))
plt.bar(difficulty_counts.keys(), difficulty_counts.values())
plt.xticks(rotation=45, ha="right")
plt.xlabel("Difficulty Level")
plt.ylabel("Frequency")
plt.title("Distribution of Problem Difficulties")
plt.show()
traced_executions = 0
traced_functions = 0
traced_blocks = 0
traced_lines = 0
traced_functions_per_solution = 0
for problem_idx, problem_data in enumerate(parser):
    if target_difficulties is not None and problem_data["difficulty"] not in target_difficulties:
        continue
    traced_executions += len(problem_data["trace_results"])
    for _, trace_res in problem_data["trace_results"].items():
        trace_res = pyine.utils.code_exec.TraceResult(**trace_res)
        traced_steps = [t for t in trace_res.traced_steps if t is not None]
        traced_source_lines: set[int] = set()
        for t in traced_steps:
            if t.trace_key.file == pyine.utils.code_exec.EXEC_TRACE_FILE_NAME:
                traced_source_lines.add(t.trace_key.line)
        traced_lines += len(traced_source_lines)
        for code_block_start_line, code_block_data in trace_res.code_blocks.items():
            if code_block_start_line not in traced_source_lines:
                continue
            traced_blocks += 1
            if code_block_data.type == pyine.utils.code_blocks.CodeBlockType.FUNCTION:
                traced_functions += 1
print(f"{traced_executions=}")
print(f"{traced_functions=}")
print(f"{traced_blocks=}")
print(f"{traced_lines=}")

In [ ]:
def _filter_relevant_source_trace_step_idxs(
    trace_res: pyine.utils.code_exec.TraceResult,
    first_relevant_step_idx: int = 0,
) -> list[int]:
    """Filters out irrelevant trace steps that are invalid or outside the proposed code string."""
    raw_trace_steps: list[pyine.utils.code_exec.TraceEvent | None] = trace_res.traced_steps
    filtered_trace_step_idxs = []
    for trace_step in raw_trace_steps:
        if trace_step is None:
            continue  # step originates from blacklisted, internal, or compiled modules
        if trace_step.trace_step_idx < first_relevant_step_idx:
            continue  # trace step occurs before we begin tracing the actual algo exec
        trace_key = trace_step.trace_key
        if trace_key.file != pyine.utils.code_exec.EXEC_TRACE_FILE_NAME:
            continue  # step originates from a separate file instead of the input code string
        # if trace_step.event_type == "call" and trace_step.trace_step_idx == first_relevant_step_idx:
        #     continue  # step is the initial call of the algo execution (useless?)
        filtered_trace_step_idxs.append(trace_step.trace_step_idx)
    return filtered_trace_step_idxs


class EventRelationship(enum.StrEnum):
    """Types of relationships between consecutive trace events."""

    # @@@@ TODO: add control block relationships as well? (or as a STEP_INTO interpretation option when identifying relationships?)
    ENTRYPOINT = enum.auto()
    STEP_OVER = enum.auto()
    STEP_INTO = enum.auto()
    STEP_OUT = enum.auto()
    RAISE = enum.auto()
    PROPAGATE = enum.auto()
    UNKNOWN = enum.auto()


def _get_relation_between_events(
    curr: pyine.utils.code_exec.TraceEvent,
    next: pyine.utils.code_exec.TraceEvent,
) -> EventRelationship:
    """Returns the relation between two trace events."""
    assert curr.trace_step_idx < next.trace_step_idx, "out-of-order trace events?"
    assert (
        curr.event_type != pyine.utils.code_exec.TraceEventType.RETURN
    ), "return events should not precede other events"
    if (
        curr.event_type == pyine.utils.code_exec.TraceEventType.LINE
        and next.event_type == pyine.utils.code_exec.TraceEventType.LINE
    ):
        return EventRelationship.STEP_OVER
    if curr.event_type == pyine.utils.code_exec.TraceEventType.CALL:
        return EventRelationship.ENTRYPOINT  # there should only ever be one of these
    if next.event_type == pyine.utils.code_exec.TraceEventType.CALL:
        assert curr.event_type == pyine.utils.code_exec.TraceEventType.LINE
        return EventRelationship.STEP_INTO
    if next.event_type == pyine.utils.code_exec.TraceEventType.RETURN:
        assert curr.event_type in [
            pyine.utils.code_exec.TraceEventType.LINE,
            pyine.utils.code_exec.TraceEventType.RETURN,
        ]
        return EventRelationship.STEP_OUT
    if (
        curr.event_type != pyine.utils.code_exec.TraceEventType.EXCEPTION
        and next.event_type == pyine.utils.code_exec.TraceEventType.EXCEPTION
    ):
        return EventRelationship.RAISE
    if (
        curr.event_type == pyine.utils.code_exec.TraceEventType.EXCEPTION
        and next.event_type != pyine.utils.code_exec.TraceEventType.EXCEPTION
    ):
        return EventRelationship.PROPAGATE
    return EventRelationship.UNKNOWN


@dataclasses.dataclass(frozen=True)
class TraceDelta:
    curr_trace_key: pyine.utils.code_exec.TraceKey
    """The trace key associated with the start of the delta."""
    next_trace_key: pyine.utils.code_exec.TraceKey
    """The trace key associated with the end of the delta."""
    trace_step_idx: int
    """The trace step index at the start of the delta; should be unique for each delta."""
    exception: dict[str, typing.Any] | None
    """A dictionary containing information about the exception being raised/propagated, if any."""
    variables_delta: dict[str, str]
    """A dictionary containing the added/removed/updated variables in the delta."""
    event_relationship: EventRelationship
    """The relationship between the start/end trace events."""

    def __repr__(self):
        """Returns a string representation of the trace delta."""
        if (
            self.curr_trace_key.file == pyine.utils.code_exec.EXEC_TRACE_FILE_NAME
            and self.next_trace_key.file == pyine.utils.code_exec.EXEC_TRACE_FILE_NAME
        ):
            return (
                f"L{self.curr_trace_key.line} -> L{self.next_trace_key.line} "
                f"({self.event_relationship}) @ step#{self.trace_step_idx} "
                f" = {self.variables_delta}"
            )
        else:
            return (
                f"{self.curr_trace_key.file}:L{self.curr_trace_key.line} -> {self.next_trace_key.file}:L{self.next_trace_key.line} "
                f"({self.event_relationship}) @ step#{self.trace_step_idx} "
                f" = {self.variables_delta}"
            )

    curr_step: pyine.utils.code_exec.TraceEvent | None = None
    """The current trace event (FOR DEBUGGING PURPOSES ONLY, DO NOT USE IN PRODUCTION CODE)."""
    next_step: pyine.utils.code_exec.TraceEvent | None = None
    """The next trace event (FOR DEBUGGING PURPOSES ONLY, DO NOT USE IN PRODUCTION CODE)."""


class DeltaGeneratorType(enum.StrEnum):
    """Supported variable state delta generation approaches."""

    SIMPLE = enum.auto()
    DEEPDIFF = enum.auto()


def simple_delta_generator(curr: dict[str, str], next: dict[str, str]) -> dict[str, str]:
    """Compares two variable dicts and returns added/updated entries."""
    # note: we purposefully do NOT show missing/removed values in deltas to reduce useless spam/outputs
    # @@@@@@ TODO: figure out if we should also add the delta of stderr/stdout?
    output = {}
    for key in next.keys() - curr.keys():  # keys added to next
        output[key] = next[key]
    for key in curr.keys() & next.keys():  # keys updated in next
        if curr[key] != next[key]:
            output[key] = next[key]
    return output


def _get_deltas_from_trace_steps(
    relevant_step_idxs: list[pyine.utils.code_exec.TraceEvent],
    trace_res: pyine.utils.code_exec.TraceResult,
    delta_generator: DeltaGeneratorType,
) -> list[TraceDelta]:
    assert len(relevant_step_idxs) > 1
    assert delta_generator in DeltaGeneratorType
    if delta_generator == DeltaGeneratorType.SIMPLE:
        delta_generator = simple_delta_generator
    else:
        delta_generator = deepdiff.DeepDiff
    output_deltas = []
    iter_idx = 0
    call_vars_stack = []
    while iter_idx < len(relevant_step_idxs) - 1:
        curr_step_idx = relevant_step_idxs[iter_idx]
        curr_step: pyine.utils.code_exec.TraceEvent = trace_res.traced_steps[curr_step_idx]
        assert curr_step is not None and curr_step.trace_step_idx == curr_step_idx
        assert curr_step.trace_key.file == pyine.utils.code_exec.EXEC_TRACE_FILE_NAME
        next_step_idx = relevant_step_idxs[iter_idx + 1]
        next_step: pyine.utils.code_exec.TraceEvent = trace_res.traced_steps[next_step_idx]
        assert next_step is not None and next_step.trace_step_idx > curr_step_idx
        assert next_step.trace_key.file == pyine.utils.code_exec.EXEC_TRACE_FILE_NAME
        assert curr_step.trace_step_idx < next_step.trace_step_idx
        relationship = _get_relation_between_events(curr_step, next_step)
        delta = {}
        if relationship == EventRelationship.ENTRYPOINT:
            call_vars_stack.append(
                (
                    pyine.utils.code_exec.TraceKey(
                        # fill these with arbitrary but unique values to be able to easily identify it
                        file=pyine.utils.code_exec.EXEC_PARENT_FILE_NAME,
                        line=-1,
                        object="<module>",
                    ),
                    curr_step.arguments,
                )
            )
            iter_idx += 1
        elif relationship == EventRelationship.STEP_OUT:
            # decompose this specific event into two deltas, to make sure we capture everything
            delta["__return__"] = next_step.return_value
            caller_trace_key, caller_vars = call_vars_stack.pop()
            if caller_trace_key.file == pyine.utils.code_exec.EXEC_PARENT_FILE_NAME:
                assert len(call_vars_stack) == 0 and len(next_step.stack_trace) == 1
            else:
                assert caller_trace_key == next_step.stack_trace[1]
            output_deltas.append(
                TraceDelta(
                    curr_trace_key=curr_step.trace_key,
                    next_trace_key=caller_trace_key,
                    trace_step_idx=curr_step.trace_step_idx,
                    exception=curr_step.exception,
                    variables_delta=delta,
                    event_relationship=relationship,
                    curr_step=curr_step,  # GET RID OF ME @@@@
                    next_step=next_step,  # GET RID OF ME @@@@
                )
            )
            iter_idx += 1
            if iter_idx < len(relevant_step_idxs) - 1:
                # this is not the final return call, so add another STEP_OVER delta to capture the next line change
                next_next_step_idx = relevant_step_idxs[iter_idx + 1]
                next_next_step: pyine.utils.code_exec.TraceEvent = trace_res.traced_steps[next_next_step_idx]
                assert next_next_step is not None and next_next_step.trace_step_idx > next_step_idx
                assert next_next_step.trace_key.file == pyine.utils.code_exec.EXEC_TRACE_FILE_NAME
                assert curr_step.trace_step_idx < next_next_step.trace_step_idx
                delta = delta_generator(caller_vars, next_next_step.variables)
                output_deltas.append(
                    TraceDelta(
                        curr_trace_key=caller_trace_key,
                        next_trace_key=next_next_step.trace_key,
                        trace_step_idx=next_step.trace_step_idx,
                        exception=next_step.exception,
                        variables_delta=delta,
                        event_relationship=EventRelationship.STEP_OVER,
                        curr_step=next_step,  # GET RID OF ME @@@@
                        next_step=next_next_step,  # GET RID OF ME @@@@
                    )
                )
                iter_idx += 1
        else:
            # all cases herein only produce a single delta that captures all potential information
            if relationship == EventRelationship.STEP_OVER:
                delta = delta_generator(curr_step.variables, next_step.variables)
            elif relationship == EventRelationship.STEP_INTO:
                delta["__call__"] = next_step.trace_key.object
                delta["__args__"] = next_step.arguments
                call_vars_stack.append((curr_step.trace_key, curr_step.variables))
                iter_idx += 1  # there will be a useless trace event following all calls, skip it?
            elif relationship == EventRelationship.RAISE or relationship == EventRelationship.PROPAGATE:
                # @@@@@ TODO: do something with call vars stack?
                delta["__exception__"] = next_step.exception._asdict()  # noqa
            else:
                print("wtf?")
            output_deltas.append(
                TraceDelta(
                    curr_trace_key=curr_step.trace_key,
                    next_trace_key=next_step.trace_key,
                    trace_step_idx=curr_step.trace_step_idx,
                    exception=next_step.exception,
                    variables_delta=delta,
                    event_relationship=relationship,
                    curr_step=curr_step,  # GET RID OF ME @@@@
                    next_step=next_step,  # GET RID OF ME @@@@
                )
            )
            iter_idx += 1
    return output_deltas


delta_generator = DeltaGeneratorType.SIMPLE

for problem_idx, problem_data in enumerate(parser):
    if target_difficulties is not None and problem_data["difficulty"] not in target_difficulties:
        continue
    for trace_id, trace_res in problem_data["trace_results"].items():
        trace_id = pyine.utils.code_exec.TraceResultIdentifier(**dict(trace_id))
        trace_res = pyine.utils.code_exec.TraceResult(**trace_res)
        if trace_id.sample_idx == 13856 and trace_id.version_idx == 0:
            continue  # annoying lambda example
        first_relevant_step_idx = 0
        if trace_res.entrypoint_step_idx is not None:
            first_entrypoint_trace_step = next(
                (v for v in trace_res.traced_steps[trace_res.entrypoint_step_idx :] if v is not None),
                None,
            )
            if first_entrypoint_trace_step is None:
                continue  # invalid entrypoint call? (might want to log/fix these?)  @@@@@
            first_relevant_step_idx = first_entrypoint_trace_step.trace_step_idx

        code_string = trace_res.code_string
        print("CODE:")
        pyine.utils.portability.print_code_with_numbered_lines(code_string)
        relevant_traced_step_idxs = _filter_relevant_source_trace_step_idxs(trace_res, first_relevant_step_idx)
        deltas = _get_deltas_from_trace_steps(relevant_traced_step_idxs, trace_res, delta_generator)
        print("DELTAS:")
        for delta_idx, delta in enumerate(deltas):
            print(f"\td#{delta_idx}:\t{delta}")
        a = 1